Yuda, IF403, 240401010353

In [35]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [36]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']
# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
 n_item = np.random.randint(2, 6)
 transaksi.append(list(np.random.choice(produk, n_item, replace=False)))
# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
 if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
  transaksi[i].append('Selai')
print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))


Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [37]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())


    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [38]:
from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
 freq = apriori(df, min_support=ms, use_colnames=True)
 print(f'min_support={ms}: {len(freq)} itemset ditemukan')
# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


In [39]:
from mlxtend.frequent_patterns import association_rules
rules = association_rules(freq_items, metric='confidence',
 min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents',
 'support', 'confidence', 'lift']].head(10))


         antecedents consequents  support  confidence      lift
8        (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
13  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
11      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
9       (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
14     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
10     (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
12     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
15   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


### Interpretasi Aturan Asosiasi (Langkah 4)

1. **Aturan yang paling kuat (Lift tertinggi):**
   Aturan yang paling kuat adalah **{Teh, Keju} $\rightarrow$ {Telur}** dengan nilai **Lift mencapai 2.38** (dan tingkat *Confidence* 85.7%). Ini menunjukkan bahwa pelanggan yang membeli Teh dan Keju bersamaan memiliki kemungkinan hampir 2.4 kali lebih besar untuk juga membeli Telur dibandingkan jika mereka membeli Telur secara acak.

2. **Apakah masuk akal secara bisnis?**
   * **Sangat Masuk Akal:** Aturan seperti **{Roti} $\rightarrow$ {Selai}** (Lift: 1.32, Confidence: 68.7%) dan **{Gula, Roti} $\rightarrow$ {Selai}** (Lift: 1.92, Confidence: 100%) sangat masuk akal secara bisnis karena merupakan kombinasi produk komplementer yang biasa dikonsumsi bersamaan saat sarapan.
   * **Pola Tersembunyi:** Kombinasi seperti **{Teh, Keju} $\rightarrow$ {Telur}** mungkin terlihat tidak biasa secara langsung, namun dalam bisnis retail, pola kuliner lokal atau kebiasaan belanja mingguan (stok bahan makanan pokok) sering kali memicu kemunculan kombinasi ini.
   * **Aksi Bisnis:** Hasil ini dapat digunakan untuk strategi *product bundling* sarapan (Roti + Selai) atau menempatkan rak produk Teh, Keju, dan Telur di area yang strategis untuk mempermudah konsumen.

In [40]:
from sklearn.metrics.pairwise import cosine_similarity
katalog = pd.DataFrame({
 'produk': produk,
 'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

In [41]:
def rekomendasi_serupa(nama_produk, top_n=3):
 idx = katalog.index[katalog['produk'] == nama_produk][0]
 skor = list(enumerate(sim_matrix[idx]))
 skor = sorted(skor, key=lambda x: x[1], reverse=True)
 skor = [s for s in skor if s[0] != idx][:top_n]
 return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))


Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [42]:
produk_target = 'Roti'
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung
produk_target
rules_terkait = rules[rules['antecedents'].apply(
 lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


### Diskusi Perbandingan Pendekatan (Langkah 6)

1. **Apakah kedua pendekatan memberikan rekomendasi yang konsisten?**
   * **Tidak selalu konsisten, namun saling melengkapi.**
   * **Association Rules** (Langkah 4) murni berbasis pada statistik transaksi aktual (*co-occurrence*). Pendekatan ini menemukan bahwa pelanggan yang membeli `Roti` sangat sering membeli `Selai` karena pola belanja nyata yang disuntikkan ke dalam dataset.
   * **Content-Based Filtering** (Langkah 5) memberikan rekomendasi berdasarkan kemiripan teks/atribut produk (menggunakan *Cosine Similarity* pada nama produk). Hasilnya bisa jadi berbeda karena ia mencari kata atau karakteristik yang mirip, bukan berdasarkan riwayat transaksi.
   * **Kesimpulan:** Keduanya bisa berbeda karena yang satu melihat **perilaku belanja (behavior)**, sedangkan yang lain melihat **karakteristik produk (content)**.

2. **Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (Hybrid)?**

   * **Gunakan Association Rules saja jika:**
     * Anda memiliki data transaksi historis yang sangat besar dan ingin melakukan strategi fisik seperti penataan rak supermarket (*Market Basket Analysis*) atau paket *bundling* produk (misal: paket hemat sarapan).
   
   * **Gunakan Content-Based Filtering saja jika:**
     * Menghadapi masalah *Cold Start* untuk **produk baru**. Produk baru belum memiliki riwayat transaksi sama sekali (sehingga tidak bisa dideteksi oleh Association Rules), namun kita tetap bisa merekomendasikannya kepada pengguna berdasarkan deskripsi atau kategorinya yang mirip dengan produk lama.

   * **Kapan sebaiknya menggabungkan keduanya (Hybrid)?**
     * **Sistem E-commerce Modern:** Sangat disarankan menggabungkan keduanya dalam sistem *Hybrid*.
     * **Skenario Aplikasi:** Anda bisa menggunakan *Association Rules* untuk merekomendasikan produk pelengkap di halaman *checkout* (misal: "Pengendara yang membeli kamera ini juga membeli kartu memori"). Di sisi lain, Anda menggunakan *Content-Based* di halaman utama untuk menampilkan variasi produk yang serupa (misal: "Kamera lain yang mirip dengan yang Anda lihat"). Penggabungan ini menghasilkan sistem rekomendasi yang jauh lebih cerdas, relevan, dan fleksibel.